In [ ]:
import os


os.environ.setdefault("HF_HOME", "/nfsd/lttm4/tesisti/shahrampour/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_transformers")

for k in ["HF_HOME","HF_DATASETS_CACHE","TRANSFORMERS_CACHE"]:
    os.makedirs(os.environ[k], exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])

## 1) Imports

In [ ]:
import numpy as np
import torch
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
import copy
import torch.nn.functional as F

from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from safetensors.torch import load_file as safe_load_file
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import set_seed

from peft import LoraConfig, get_peft_model, TaskType

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
RUN_NAME = "cifar100_5step_e15_10_5_15_V5.2"

RESULTS_DIR = os.path.join("results", RUN_NAME)
PLOTS_DIR   = os.path.join(RESULTS_DIR, "plots")
TABLES_DIR  = os.path.join(RESULTS_DIR, "tables")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")
OUTPUTS_DIR = os.path.join("outputs", RUN_NAME)

STEP1_FINAL_OUT = os.path.join(OUTPUTS_DIR, "step1_scratch_final")
RANK_SWEEP_DIR  = os.path.join(OUTPUTS_DIR, "rank_sweep")
MERGE_DIR       = os.path.join(OUTPUTS_DIR, "merged_across_ranks")

for d in [RESULTS_DIR, PLOTS_DIR, TABLES_DIR, METRICS_DIR, OUTPUTS_DIR, STEP1_FINAL_OUT, RANK_SWEEP_DIR, MERGE_DIR]:
    os.makedirs(d, exist_ok=True)

## 2) Load CIFAR-100 (fine labels)

In [ ]:
from datasets import Image

dataset = load_dataset("cifar100")
dataset = dataset.rename_column("fine_label", "label")

dataset = dataset.cast_column("img", Image())

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

print("num_classes:", num_classes)
print("example keys:", dataset["train"][0].keys())
print("first 10 classes:", class_names[:10])

In [ ]:
def make_custom_eval_dataset(class_ids, remap_labels=True):
    test_ds = filter_dataset_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        test_ds = test_ds.map(remap)
    else:
        label2new = None
        new2orig = None

    test_ds.set_transform(preprocess_val)
    return test_ds, label2new, new2orig

## 3) Define incremental class splits (2/5/10 steps)

In [ ]:

num_steps = 5  

assert num_classes % num_steps == 0
classes_per_step = num_classes // num_steps

class_splits = [
    list(range(s * classes_per_step, (s + 1) * classes_per_step))
    for s in range(num_steps)
]

print("classes per step:", classes_per_step)
print("split sizes:", [len(x) for x in class_splits])
print("step0 sample:", class_splits[0][:10], "...", class_splits[0][-3:])

In [ ]:
first_step_classes = class_splits[0]
later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(class_splits[s])
all_classes = list(range(num_classes))

print("first step classes:", len(first_step_classes))
print("later step classes:", len(later_step_classes))
print("all classes:", len(all_classes))

## 4) Model + preprocessing

In [ ]:
# Requested model
model_checkpoint = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

from torchvision import transforms
from PIL import Image
import numpy as np
import torch

size = image_processor.size
if isinstance(size, dict):
    H = size.get("height", 224)
    W = size.get("width", 224)
else:
    H = W = int(size)

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = x.astype(np.uint8)
        arr = np.squeeze(arr)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if not (arr.ndim == 3 and arr.shape[-1] in (1, 3)):
            raise TypeError(f"Unexpected image array shape after squeeze: {arr.shape}")

        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean().item()}

## 5) Build per-step datasets (step / cumulative / full)

In [ ]:
def classes_for_step(step_idx: int) -> list[int]:
    return class_splits[step_idx]

def classes_for_cumulative(step_idx: int) -> list[int]:
    out = []
    for s in range(step_idx + 1):
        out.extend(class_splits[s])
    return out

def filter_by_classes(ds, class_ids):
    class_set = set(class_ids)
    ds = ds.with_format(None)
    return ds.filter(lambda x: int(x["label"]) in class_set)

def make_step_datasets(step_idx: int, split_type: str = "new_only", remap_labels: bool = False):
    """
    split_type:
      - 'new_only'   : only classes of this step
      - 'cumulative' : union of classes up to this step
      - 'full'       : all classes (100)
    """
    if split_type == "full":
        class_ids = list(range(num_classes))
    elif split_type == "cumulative":
        class_ids = classes_for_cumulative(step_idx)
    elif split_type == "new_only":
        class_ids = classes_for_step(step_idx)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")

    train_ds = filter_by_classes(dataset["train"], class_ids)
    test_ds  = filter_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        train_ds = train_ds.map(remap)
        test_ds  = test_ds.map(remap)
    else:
        label2new = {c: c for c in class_ids}
        new2orig = {c: c for c in class_ids}

    train_ds = train_ds.with_transform(preprocess_train)
    test_ds = test_ds.with_transform(preprocess_val)

    print(f"[Dataset] Step {step_idx+1} | split_type={split_type}")
    print(f"[Dataset] Classes: {class_ids[:5]} ... {class_ids[-5:]}")
    print(f"[Dataset] Train size: {len(train_ds)} | Test size: {len(test_ds)}")

    return train_ds, test_ds, label2new, new2orig, class_ids

eval_all = dataset["test"].with_transform(preprocess_val)

print("eval_all size:", len(eval_all))

In [ ]:
from datasets import concatenate_datasets

def sample_replay_examples(ds, class_ids, per_class=50, seed=42):
    pieces = []
    for c in class_ids:
        sub = ds.filter(lambda x: int(x["label"]) == int(c))
        if len(sub) == 0:
            continue
        take_n = min(per_class, len(sub))
        sub = sub.shuffle(seed=seed + int(c)).select(range(take_n))
        pieces.append(sub)

    if len(pieces) == 0:
        return None

    return concatenate_datasets(pieces)


def make_step_train_with_replay(step_idx, per_class=50, seed=42, train_transform=None):
    current_classes = classes_for_step(step_idx)
    train_new = dataset["train"].filter(lambda x: int(x["label"]) in current_classes)

    old_classes = []
    if step_idx > 0:
        old_classes = classes_for_cumulative(step_idx - 1)

    if len(old_classes) == 0:
        out = train_new.shuffle(seed=seed + step_idx)
        if train_transform is not None:
            out = out.with_transform(train_transform)
        return out, current_classes, old_classes

    replay_ds = sample_replay_examples(
        dataset["train"],
        class_ids=old_classes,
        per_class=per_class,
        seed=seed + step_idx,
    )

    if replay_ds is None:
        out = train_new.shuffle(seed=seed + step_idx)
        if train_transform is not None:
            out = out.with_transform(train_transform)
        return out, current_classes, old_classes

    train_mix = concatenate_datasets([train_new, replay_ds]).shuffle(seed=seed + step_idx)

    if train_transform is not None:
        train_mix = train_mix.with_transform(train_transform)

    return train_mix, current_classes, old_classes

## 6) Training recipes (reasonable settings)

In [ ]:
set_seed(42)

SCRATCH_EPOCHS = 1
LORA_EPOCHS    = 1
FT_EPOCHS      = 1
JOINT_EPOCHS   = 1

# SCRATCH_EPOCHS = 15
# LORA_EPOCHS    = 6
# FT_EPOCHS      = 5
# JOINT_EPOCHS   = 15

USE_REPLAY = True
REPLAY_PER_CLASS = 150
REPLAY_SEED = 42

BATCH_SCRATCH = 8
ACCUM_SCRATCH = 2

BATCH_LORA    = 16
ACCUM_LORA    = 1

BATCH_FT      = 8
ACCUM_FT      = 2

LR_SCRATCH = 5e-5
LR_LORA    = 1e-4
LR_FT      = 3e-6
LR_JOINT   = 5e-5

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# فقط اینها مهم‌اند
RANK_LIST = [8, 16, 32]
LORA_ALPHA_MULTIPLIER = 2
LORA_DROPOUT = 0.10
LORA_TARGET_MODULES = ["query", "key", "value"]
LORA_MODULES_TO_SAVE = ["classifier"]
LORA_LABEL_SMOOTHING = 0.05
LORA_MONITOR_SET = "all_seen"

results = []
rank_lora_results = []
merged_rank_results = []

In [ ]:
run_config = {
    "run_name": RUN_NAME,
    "model_checkpoint": model_checkpoint,
    "init_mode": "scratch",
    "num_classes": num_classes,
    "num_steps": num_steps,
    "classes_per_step": classes_per_step,
    "scratch_epochs": SCRATCH_EPOCHS,
    "lora_epochs": LORA_EPOCHS,
    "ft_epochs": FT_EPOCHS,
    "joint_epochs": JOINT_EPOCHS,
    "lr_scratch": LR_SCRATCH,
    "lr_lora": LR_LORA,
    "lr_ft": LR_FT,
    "lr_joint": LR_JOINT,
    "use_replay": USE_REPLAY,
    "replay_per_class": REPLAY_PER_CLASS,
    "rank_list": RANK_LIST,
    "lora_alpha_multiplier": LORA_ALPHA_MULTIPLIER,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "lora_modules_to_save": LORA_MODULES_TO_SAVE,
    "lora_label_smoothing": LORA_LABEL_SMOOTHING,
    "lora_monitor_set": LORA_MONITOR_SET,
}

with open(os.path.join(METRICS_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

## 7) Step 1: train full ViT from scratch on step 0 classes

In [ ]:
import os, shutil

# --- clean old step1 outputs so stale 20-class checkpoints cannot survive ---
if os.path.exists(STEP1_FINAL_OUT):
    shutil.rmtree(STEP1_FINAL_OUT)
if os.path.exists(STEP1_FINAL_OUT):
    shutil.rmtree(STEP1_FINAL_OUT)

step1_idx = 0
train_step1, test_step1, label2new_1, new2orig_1, class_ids_1 = make_step_datasets(
    step1_idx, split_type="new_only", remap_labels=False
)

print("Step1 original classes:", class_ids_1[:5], "...", class_ids_1[-5:])
print("Step1 label range:",
      min(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))),
      max(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))))

config_step1 = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

model_step1 = AutoModelForImageClassification.from_config(config_step1)

print("Before training - Step1 model num_labels:", model_step1.config.num_labels)
print("Before training - Step1 classifier out_features:", model_step1.classifier.out_features)
assert model_step1.config.num_labels == num_classes
assert model_step1.classifier.out_features == num_classes

args_step1 = TrainingArguments(
    output_dir=STEP1_FINAL_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=SCRATCH_EPOCHS,
    learning_rate=LR_SCRATCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_SCRATCH,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_SCRATCH,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
    max_grad_norm=1.0,
)

trainer_step1 = Trainer(
    model=model_step1,
    args=args_step1,
    train_dataset=train_step1,
    eval_dataset=test_step1,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback
trainer_step1.remove_callback(NotebookProgressCallback)

trainer_step1.train()

eval_step1 = trainer_step1.evaluate()
print({"eval_step1": eval_step1})

trainer_step1.save_model(STEP1_FINAL_OUT)
image_processor.save_pretrained(STEP1_FINAL_OUT)


reloaded_step1 = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)
print("Reloaded STEP1 num_labels:", reloaded_step1.config.num_labels)
print("Reloaded STEP1 classifier out_features:", reloaded_step1.classifier.out_features)

assert reloaded_step1.config.num_labels == num_classes
assert reloaded_step1.classifier.out_features == num_classes

In [ ]:
print("Init mode:", run_config["init_mode"])
assert run_config["init_mode"] == "scratch"

In [ ]:
step1_log_df = pd.DataFrame(trainer_step1.state.log_history)
step1_log_df.to_csv(os.path.join(TABLES_DIR, "step1_log_history.csv"), index=False)

step1_metrics = {
    "experiment": "step1_scratch",
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "step1_metrics.json"), "w") as f:
    json.dump(step1_metrics, f, indent=2)

results.append({
    "experiment": "step1_scratch",
    "method": "full_finetune",
    "step": 1,
    "eval_type": "current_step",
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
})

step1_log_df.tail()

In [ ]:
if "loss" in step1_log_df.columns:
    train_loss_df = step1_log_df[step1_log_df["loss"].notna()]
    if not train_loss_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(train_loss_df["step"], train_loss_df["loss"])
        plt.xlabel("Step")
        plt.ylabel("Train Loss")
        plt.title("Step 1 Train Loss")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_train_loss.png"), dpi=200)
        plt.show()

if "eval_accuracy" in step1_log_df.columns:
    eval_df = step1_log_df[step1_log_df["eval_accuracy"].notna()]
    if not eval_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(eval_df["epoch"], eval_df["eval_accuracy"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Eval Accuracy")
        plt.title("Step 1 Eval Accuracy")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_eval_accuracy.png"), dpi=200)
        plt.show()

## 8) Step 2: LoRA only (freeze backbone) on top of Step 1 model

In [ ]:
first_step_classes = classes_for_step(0)

def make_eval_dataset(class_ids, name=None):
    test_ds = filter_by_classes(dataset["test"], class_ids)
    test_ds = test_ds.with_transform(preprocess_val)
    if name is not None:
        print(f"[Eval set] {name}: num_classes={len(class_ids)}, size={len(test_ds)}")
    return test_ds

In [ ]:
import os, shutil
from peft import LoraConfig, get_peft_model, PeftModel

first_step_classes = classes_for_step(0)

def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)

def run_lora_pipeline_for_rank(rank_value: int):
    rank_results = []
    rank_dir = os.path.join(RANK_SWEEP_DIR, f"rank_{rank_value}")
    os.makedirs(rank_dir, exist_ok=True)

   
    for step_idx in range(1, num_steps):
        stale_train_dir = os.path.join(rank_dir, f"step_{step_idx+1}_train")
        stale_adapter_dir = os.path.join(rank_dir, f"step_{step_idx+1}_adapter")
        if os.path.exists(stale_train_dir):
            shutil.rmtree(stale_train_dir)
        if os.path.exists(stale_adapter_dir):
            shutil.rmtree(stale_adapter_dir)

    base_model_dir = STEP1_FINAL_OUT
    

    for step_idx in range(1, num_steps):
        if USE_REPLAY:
            train_step, class_ids, old_classes = make_step_train_with_replay(
                step_idx=step_idx,
                per_class=REPLAY_PER_CLASS,
                seed=REPLAY_SEED,
                train_transform=preprocess_train,
            )

            _, test_step, label2new, new2orig, _ = make_step_datasets(
                step_idx,
                split_type="new_only",
                remap_labels=False,
            )

            print(f"\n[LoRA+Replay | r={rank_value}] Step {step_idx+1}")
            print("Current step classes:", class_ids[:5], "...", class_ids[-5:])
            print("Replay old classes count:", len(old_classes))
            print("Train size with replay:", len(train_step))
        else:
            train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
                step_idx,
                split_type="new_only",
                remap_labels=False,
            )

            print(f"\n[LoRA separate | r={rank_value}] Step {step_idx+1}")
            print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

        tr_min, tr_max = _label_range(train_step)
        te_min, te_max = _label_range(test_step)

        print("Train label range:", tr_min, tr_max)
        print("Test label range:", te_min, te_max)
        print("Num labels:", num_classes)

        seen_now = classes_for_cumulative(step_idx)
        later_now = [c for c in seen_now if c not in classes_for_step(0)]

        eval_current = test_step
        eval_first = make_eval_dataset(classes_for_step(0), name=f"r{rank_value}_step{step_idx+1}_first_step")
        eval_later = make_eval_dataset(later_now, name=f"r{rank_value}_step{step_idx+1}_later_steps") if len(later_now) > 0 else None
        eval_seen  = make_eval_dataset(seen_now, name=f"r{rank_value}_step{step_idx+1}_all_seen")

        if LORA_MONITOR_SET == "all_seen":
            eval_monitor = eval_seen
            monitor_name = "all_seen"
        else:
            eval_monitor = eval_current
            monitor_name = "current_step"

        print("Loaded fixed base_model_dir:", base_model_dir)
        base_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

        assert base_model.config.num_labels == num_classes
        assert base_model.classifier.out_features == num_classes

        lora_cfg = LoraConfig(
            r=rank_value,
            lora_alpha=rank_value * LORA_ALPHA_MULTIPLIER,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            target_modules=LORA_TARGET_MODULES,
            modules_to_save=LORA_MODULES_TO_SAVE,
            task_type="IMAGE_CLASSIFICATION",
        )

        model_lora = get_peft_model(base_model, lora_cfg)
        model_lora.print_trainable_parameters()

        out_dir = os.path.join(rank_dir, f"step_{step_idx+1}_train")

        trainer_lora = Trainer(
            model=model_lora,
            args=TrainingArguments(
                output_dir=out_dir,
                remove_unused_columns=False,
                per_device_train_batch_size=BATCH_LORA,
                per_device_eval_batch_size=BATCH_LORA,
                gradient_accumulation_steps=ACCUM_LORA,
                num_train_epochs=LORA_EPOCHS,
                learning_rate=LR_LORA,
                weight_decay=WEIGHT_DECAY,
                warmup_ratio=WARMUP_RATIO,
                lr_scheduler_type=SCHED,
                fp16=USE_FP16,
                evaluation_strategy="epoch",
                save_strategy="epoch",
                logging_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model="eval_accuracy",
                greater_is_better=True,
                save_total_limit=1,
                report_to="none",
                label_smoothing_factor=LORA_LABEL_SMOOTHING,
            ),
            train_dataset=train_step,
            eval_dataset=eval_monitor,
            data_collator=collate_fn,
            compute_metrics=compute_metrics,
            tokenizer=image_processor,
        )

        trainer_lora.train()

        eval_current_res = trainer_lora.evaluate(eval_dataset=eval_current)
        eval_first_res   = trainer_lora.evaluate(eval_dataset=eval_first)
        eval_later_res   = trainer_lora.evaluate(eval_dataset=eval_later) if eval_later is not None else {"eval_accuracy": np.nan, "eval_loss": np.nan}
        eval_seen_res    = trainer_lora.evaluate(eval_dataset=eval_seen)

        step_adapter_dir = os.path.join(rank_dir, f"step_{step_idx+1}_adapter")
        trainer_lora.model.save_pretrained(step_adapter_dir)
        image_processor.save_pretrained(step_adapter_dir)


        # ---- merge this step into backbone, so next step is truly sequential ----
        merged_step_dir = os.path.join(rank_dir, f"step_{step_idx+1}_merged_model")

        merged_backbone = trainer_lora.model.merge_and_unload()
        merged_backbone.save_pretrained(merged_step_dir)
        image_processor.save_pretrained(merged_step_dir)

        # next step must start from this merged model, not from STEP1_FINAL_OUT
        base_model_dir = merged_step_dir

        adapter_meta = {
            "rank": rank_value,
            "step": step_idx + 1,
            "base_model_dir": STEP1_FINAL_OUT,
            "class_ids": class_ids,
            "num_labels": num_classes,
            "monitor_set": monitor_name,
            "replay_per_class": REPLAY_PER_CLASS if USE_REPLAY else 0,
            "lora_r": rank_value,
            "lora_alpha": rank_value * LORA_ALPHA_MULTIPLIER,
            "lora_dropout": LORA_DROPOUT,
        }
        with open(os.path.join(step_adapter_dir, "adapter_meta.json"), "w") as f:
            json.dump(adapter_meta, f, indent=2)

        rank_results.extend([
            {
                "experiment": f"rank_{rank_value}_step_{step_idx+1}",
                "method": f"lora_rank_{rank_value}",
                "rank": rank_value,
                "step": step_idx + 1,
                "eval_type": "current_step",
                "eval_accuracy": float(eval_current_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_current_res.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"rank_{rank_value}_step_{step_idx+1}",
                "method": f"lora_rank_{rank_value}",
                "rank": rank_value,
                "step": step_idx + 1,
                "eval_type": "first_step",
                "eval_accuracy": float(eval_first_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_first_res.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"rank_{rank_value}_step_{step_idx+1}",
                "method": f"lora_rank_{rank_value}",
                "rank": rank_value,
                "step": step_idx + 1,
                "eval_type": "later_steps",
                "eval_accuracy": float(eval_later_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_later_res.get("eval_loss", np.nan)),
            },
            {
                "experiment": f"rank_{rank_value}_step_{step_idx+1}",
                "method": f"lora_rank_{rank_value}",
                "rank": rank_value,
                "step": step_idx + 1,
                "eval_type": "all_seen",
                "eval_accuracy": float(eval_seen_res.get("eval_accuracy", np.nan)),
                "eval_loss": float(eval_seen_res.get("eval_loss", np.nan)),
            },
        ])

    print(f"\nFinished all steps for rank={rank_value}")
    
    final_model_dir = os.path.join(rank_dir, "final_merged_model")
    if os.path.exists(final_model_dir):
        shutil.rmtree(final_model_dir)
    shutil.copytree(base_model_dir, final_model_dir)


    return rank_results

rank_lora_results = []
for r in RANK_LIST:
    one_rank_results = run_lora_pipeline_for_rank(r)
    rank_lora_results.extend(one_rank_results)

print("\nRank sweep training finished.")

In [ ]:
from copy import deepcopy

def get_final_model_dir_for_rank(rank_value):
    return os.path.join(RANK_SWEEP_DIR, f"rank_{rank_value}", "final_merged_model")

def extract_full_model_delta_state(final_model_dir, base_model_dir=STEP1_FINAL_OUT):
    base_model = AutoModelForImageClassification.from_pretrained(base_model_dir)
    final_model = AutoModelForImageClassification.from_pretrained(final_model_dir)

    base_sd  = base_model.state_dict()
    final_sd = final_model.state_dict()

    delta_state = {}
    common_keys = set(base_sd.keys()).intersection(set(final_sd.keys()))

    for k in sorted(common_keys):
        if base_sd[k].shape == final_sd[k].shape and torch.is_floating_point(final_sd[k]):
            delta_state[k] = (final_sd[k].detach().cpu() - base_sd[k].detach().cpu())

    return delta_state

def merge_delta_states_average(rank_list):
    all_states = []

    for r in rank_list:
        final_model_dir = get_final_model_dir_for_rank(r)
        st = extract_full_model_delta_state(final_model_dir, STEP1_FINAL_OUT)
        all_states.append(st)
        print(f"Loaded full-model delta for rank={r}, tensors={len(st)}")

    common_keys = set(all_states[0].keys())
    for st in all_states[1:]:
        common_keys = common_keys.intersection(set(st.keys()))

    print("Common keys:", len(common_keys))

    merged = {}
    for k in sorted(common_keys):
        merged[k] = sum(st[k] for st in all_states) / len(all_states)

    return merged

def apply_delta_state_to_base(base_model, merged_delta_state):
    model = deepcopy(base_model)
    model_sd = model.state_dict()

    for k, delta in merged_delta_state.items():
        if k in model_sd and model_sd[k].shape == delta.shape:
            model_sd[k] = model_sd[k] + delta.to(model_sd[k].dtype)

    model.load_state_dict(model_sd, strict=False)
    return model

merged_delta_state = merge_delta_states_average(RANK_LIST)

base_for_merge = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)
merged_rank_model = apply_delta_state_to_base(base_for_merge, merged_delta_state)

os.makedirs(MERGE_DIR, exist_ok=True)
merged_rank_model.save_pretrained(MERGE_DIR)
image_processor.save_pretrained(MERGE_DIR)

eval_first_merged = make_eval_dataset(classes_for_step(0), name="merged_rank_first_step")
eval_later_merged = make_eval_dataset(later_step_classes, name="merged_rank_later_steps")
eval_seen_merged  = make_eval_dataset(all_classes, name="merged_rank_all_seen")

trainer_eval_merged = Trainer(
    model=merged_rank_model,
    args=TrainingArguments(
        output_dir=os.path.join(MERGE_DIR, "eval"),
        remove_unused_columns=False,
        per_device_eval_batch_size=BATCH_LORA,
        report_to="none",
    ),
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    tokenizer=image_processor,
)

merged_final_first = trainer_eval_merged.evaluate(eval_dataset=eval_first_merged)
merged_final_later = trainer_eval_merged.evaluate(eval_dataset=eval_later_merged)
merged_final_seen  = trainer_eval_merged.evaluate(eval_dataset=eval_seen_merged)

merged_rank_results = [
    {
        "experiment": "merged_across_ranks_final_eval",
        "method": "merged_across_ranks",
        "rank": -1,
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": float(merged_final_first.get("eval_accuracy", np.nan)),
        "eval_loss": float(merged_final_first.get("eval_loss", np.nan)),
    },
    {
        "experiment": "merged_across_ranks_final_eval",
        "method": "merged_across_ranks",
        "rank": -1,
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": float(merged_final_later.get("eval_accuracy", np.nan)),
        "eval_loss": float(merged_final_later.get("eval_loss", np.nan)),
    },
    {
        "experiment": "merged_across_ranks_final_eval",
        "method": "merged_across_ranks",
        "rank": -1,
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": float(merged_final_seen.get("eval_accuracy", np.nan)),
        "eval_loss": float(merged_final_seen.get("eval_loss", np.nan)),
    },
]

print("Merged across ranks evaluation done.")
print(merged_rank_results)

## 9) Baseline: full fine-tune instead of LoRA (Step 2)

In [ ]:
ft_results = []
base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    if USE_REPLAY:
        train_step, class_ids, old_classes = make_step_train_with_replay(
            step_idx=step_idx,
            per_class=REPLAY_PER_CLASS,
            seed=REPLAY_SEED,
            train_transform=preprocess_train,
        )

        _, test_step, label2new, new2orig, _ = make_step_datasets(
            step_idx,
            split_type="new_only",
            remap_labels=False,
        )

        print(f"\n[FT+Replay] Step {step_idx+1}")
        print("Current step classes:", class_ids[:5], "...", class_ids[-5:])
        print("Replay old classes count:", len(old_classes))
        print("Train size with replay:", len(train_step))
    else:
        train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
            step_idx,
            split_type="new_only",
            remap_labels=False,
        )

        print(f"\n[FT] Step {step_idx+1}")
        print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    ft_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

    print(f"[FT] Loaded model from {base_model_dir}")
    print(f"[FT] Step {step_idx+1} model num_labels:", ft_model.config.num_labels)
    print(f"[FT] Step {step_idx+1} classifier out_features:", ft_model.classifier.out_features)

    step_ft_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft"

    args_ft = TrainingArguments(
        output_dir=step_ft_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=FT_EPOCHS,
        learning_rate=LR_FT,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_FT,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_FT,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_ft = Trainer(
        model=ft_model,
        args=args_ft,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_ft.train()
    eval_current = trainer_ft.evaluate(test_step)

    seen_classes_now = classes_for_cumulative(step_idx)
    later_classes_now = [c for c in seen_classes_now if c not in classes_for_step(0)]

    ft_test_first = make_eval_dataset(classes_for_step(0))
    ft_test_later = make_eval_dataset(later_classes_now) if len(later_classes_now) > 0 else None
    ft_test_seen  = make_eval_dataset(seen_classes_now)

    ft_eval_first = trainer_ft.evaluate(eval_dataset=ft_test_first)
    ft_eval_later = trainer_ft.evaluate(eval_dataset=ft_test_later) if ft_test_later is not None else {"eval_accuracy": np.nan, "eval_loss": np.nan}
    ft_eval_seen  = trainer_ft.evaluate(eval_dataset=ft_test_seen)

    step_ft_final_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_final"
    os.makedirs(step_ft_final_dir, exist_ok=True)

    print(f"[FT] Step {step_idx+1} saving model to {step_ft_final_dir}")
    trainer_ft.model.save_pretrained(step_ft_final_dir, safe_serialization=False)
    image_processor.save_pretrained(step_ft_final_dir)
    print(f"[FT] Step {step_idx+1} save finished")

    reloaded_ft = AutoModelForImageClassification.from_pretrained(step_ft_final_dir)
    print(f"[FT] Reload check step {step_idx+1} num_labels:", reloaded_ft.config.num_labels)
    print(f"[FT] Reload check step {step_idx+1} classifier out_features:", reloaded_ft.classifier.out_features)

    pd.DataFrame(trainer_ft.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_ft_log_history.csv"),
        index=False
    )

    ft_results.extend([
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "ft_replay" if USE_REPLAY else "full_finetune",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "ft_replay" if USE_REPLAY else "full_finetune",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(ft_eval_first.get("eval_accuracy", np.nan)),
            "eval_loss": float(ft_eval_first.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "ft_replay" if USE_REPLAY else "full_finetune",
            "step": step_idx + 1,
            "eval_type": "later_steps_seen_so_far",
            "eval_accuracy": float(ft_eval_later.get("eval_accuracy", np.nan)),
            "eval_loss": float(ft_eval_later.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "ft_replay" if USE_REPLAY else "full_finetune",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(ft_eval_seen.get("eval_accuracy", np.nan)),
            "eval_loss": float(ft_eval_seen.get("eval_loss", np.nan)),
        },
    ])

    base_model_dir = step_ft_final_dir

print("\nFull FT continual training finished.")

In [ ]:
final_ft_dir = f"outputs/{RUN_NAME}/step_{num_steps}_ft_final"
final_ft_model = AutoModelForImageClassification.from_pretrained(final_ft_dir)

args_ft_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_ft_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_ft_final = Trainer(
    model=final_ft_model,
    args=args_ft_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

final_seen_classes = classes_for_cumulative(num_steps - 1)

ft_test_first = make_eval_dataset(first_step_classes)
ft_test_later = make_eval_dataset(later_step_classes)
ft_test_seen = make_eval_dataset(final_seen_classes)

print("Final FT num_labels:", final_ft_model.config.num_labels)
print("Final FT classifier out_features:", final_ft_model.classifier.out_features)
assert final_ft_model.classifier.out_features == num_classes

ft_final_first = trainer_ft_final.evaluate(ft_test_first)
ft_final_later = trainer_ft_final.evaluate(ft_test_later)
ft_final_seen = trainer_ft_final.evaluate(ft_test_seen)

print("FT final - first step:", ft_final_first)
print("FT final - later steps:", ft_final_later)
print("FT final - all seen:", ft_final_seen)

## 10) Upper bound: joint training (full dataset)

In [ ]:
train_joint, test_joint, label2new_J, new2orig_J, class_ids_J = make_step_datasets(
    step_idx=0, split_type="full", remap_labels=False
)

config_joint = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

joint_model = AutoModelForImageClassification.from_config(config_joint)

args_joint = TrainingArguments(
    output_dir=JOINT_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=JOINT_EPOCHS,
    learning_rate=LR_JOINT,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_FT,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_FT,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
)

trainer_joint = Trainer(
    model=joint_model,
    args=args_joint,
    train_dataset=train_joint,
    eval_dataset=test_joint,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer_joint.train()

eval_joint = trainer_joint.evaluate()
print({"eval_joint_full": eval_joint})

In [ ]:
joint_log_df = pd.DataFrame(trainer_joint.state.log_history)
joint_log_df.to_csv(os.path.join(TABLES_DIR, "joint_log_history.csv"), index=False)

joint_metrics = {
    "experiment": "joint_full",
    "eval_loss": float(eval_joint.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_joint.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "joint_metrics.json"), "w") as f:
    json.dump(joint_metrics, f, indent=2)

joint_log_df.tail()

In [ ]:
joint_test_first = make_eval_dataset(first_step_classes)
joint_test_later = make_eval_dataset(later_step_classes)
joint_test_all = make_eval_dataset(all_classes)

joint_final_first = trainer_joint.evaluate(joint_test_first)
joint_final_later = trainer_joint.evaluate(joint_test_later)
joint_final_seen = trainer_joint.evaluate(joint_test_all)

print("Joint final - first step:", joint_final_first)
print("Joint final - later steps:", joint_final_later)
print("Joint final - all seen:", joint_final_seen)

## 11) Compare results (step test vs full test)

In [ ]:
def grab_acc(d):
    return float(d["eval_accuracy"]) if "eval_accuracy" in d else np.nan

def grab_loss(d):
    return float(d["eval_loss"]) if "eval_loss" in d else np.nan

all_results = []

# B0
all_results.extend(results)

# Rank-wise LoRA runs
all_results.extend(rank_lora_results)

# Final merged-across-ranks
all_results.extend(merged_rank_results)

# Full FT
all_results.extend(ft_results)

# Joint
all_results.extend([
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "rank": -1,
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(joint_res_first),
        "eval_loss": grab_loss(joint_res_first),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "rank": -1,
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(joint_res_later),
        "eval_loss": grab_loss(joint_res_later),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "rank": -1,
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(joint_res_all),
        "eval_loss": grab_loss(joint_res_all),
    },
])

results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(TABLES_DIR, "results_summary_raw.csv"), index=False)

print(results_df.head())
print("Saved raw results.")

In [ ]:
results_df_clean = results_df.copy()

results_df_clean = results_df_clean.rename(columns={
    "eval_accuracy": "accuracy",
    "eval_loss": "loss",
    "eval_type": "eval_set"
})

def map_method(x):
    if x == "step1_scratch":
        return "B0"
    elif str(x).startswith("lora_rank_"):
        return x.replace("lora_rank_", "LoRA-r")
    elif x == "merged_across_ranks":
        return "Merged across ranks"
    elif x == "full_finetune":
        return "Full FT"
    elif x == "ft_replay":
        return "FT + Replay"
    elif x == "joint_upper_bound":
        return "Joint"
    return x

results_df_clean["method"] = results_df_clean["method"].apply(map_method)

def map_eval_stage(exp_name):
    if "final_eval" in str(exp_name):
        return "final"
    return "step"

results_df_clean["eval_stage"] = results_df_clean["experiment"].apply(map_eval_stage)

def map_adapter_name(row):
    if row["method"] == "B0":
        return "B0"
    if str(row["method"]).startswith("LoRA-r"):
        return f"r{int(row['rank'])}_s{int(row['step'])}"
    if row["method"] == "Merged across ranks":
        return "merge_ranks_avg"
    return np.nan

results_df_clean["adapter_name"] = results_df_clean.apply(map_adapter_name, axis=1)

keep_cols = ["method", "adapter_name", "rank", "step", "eval_stage", "eval_set", "accuracy", "loss"]
keep_cols = [c for c in keep_cols if c in results_df_clean.columns]

results_df_clean = results_df_clean[keep_cols].copy()
results_df_clean.to_csv(os.path.join(TABLES_DIR, "results_summary_clean.csv"), index=False)

print("Clean columns:", list(results_df_clean.columns))
print(results_df_clean)

In [ ]:
summary_lines = [
    "Experiment summary",
    "==================",
    "",
]

for _, row in results_df_clean.iterrows():
    acc = row["accuracy"]
    loss = row["loss"]

    acc_str = f"{acc:.4f}" if pd.notna(acc) else "nan"
    loss_str = f"{loss:.4f}" if pd.notna(loss) else "nan"

    summary_lines.append(
        f"method={row['method']} | adapter_name={row['adapter_name']} | "
        f"step={row['step']} | eval_stage={row['eval_stage']} | "
        f"eval_set={row['eval_set']} | accuracy={acc_str} | loss={loss_str}"
    )

with open(os.path.join(RESULTS_DIR, "summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("\n".join(summary_lines))

print("\nColumns in results_df_clean:")
print(list(results_df_clean.columns))

ft_rows = results_df_clean[
    results_df_clean["method"].astype(str).str.contains("FT|Replay", na=False)
].copy()

show_cols = [c for c in ["method", "step", "eval_stage", "eval_set", "accuracy", "loss"] if c in ft_rows.columns]

if len(ft_rows) > 0 and len(show_cols) > 0:
    print("\nFT-related rows:")
    print(
        ft_rows[show_cols].sort_values(
            [c for c in ["method", "step", "eval_stage", "eval_set"] if c in ft_rows.columns]
        )
    )
else:
    print("\nNo FT-related rows found.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_final = results_df_clean[
    (results_df_clean["eval_stage"] == "final") &
    (results_df_clean["step"] == num_steps)
].copy()

available_methods = set(df_final["method"].astype(str).unique())

if "FT + Replay" in available_methods:
    ft_name = "FT + Replay"
elif "Full FT" in available_methods:
    ft_name = "Full FT"
else:
    ft_name = None

plot_methods = [f"LoRA-r{r}" for r in RANK_LIST] + ["Merged across ranks", ft_name, "Joint"]
plot_methods = [m for m in plot_methods if m is not None]

labels = ["first_step", "later_steps", "all_seen"]

df_final = df_final[df_final["method"].isin(plot_methods)]

pivot = df_final.pivot_table(
    index="eval_set",
    columns="method",
    values="accuracy",
    aggfunc="mean"
).reindex(labels)

x = np.arange(len(labels))
width = 0.12
offsets = np.arange(-(len(plot_methods)-1)/2, (len(plot_methods)-1)/2 + 1)

plt.figure(figsize=(15, 6))

for method, offset in zip(plot_methods, offsets):
    if method in pivot.columns:
        plt.bar(x + offset * width, pivot[method], width, label=method)

plt.xticks(x, labels)
plt.ylabel("Accuracy")
plt.title("Final Accuracy Comparison: Rank Runs + Merged Across Ranks")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "final_accuracy_rank_runs_and_merge.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

In [ ]:
final_first = df_final[df_final["eval_set"] == "first_step"]
final_later = df_final[df_final["eval_set"] == "later_steps"]
final_seen  = df_final[df_final["eval_set"] == "all_seen"]

def make_barplot(df_sub, title, filename):
    methods = [f"LoRA-r{r}" for r in RANK_LIST] + ["Merged across ranks", ft_name, "Joint"]
    methods = [m for m in methods if m is not None]

    vals = []
    for m in methods:
        row = df_sub[df_sub["method"] == m]
        vals.append(float(row["accuracy"].iloc[0]) if len(row) > 0 else np.nan)

    plt.figure(figsize=(10, 4))
    plt.bar(methods, vals)
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.xticks(rotation=20)
    plt.tight_layout()

    out_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved plot to:", out_path)

make_barplot(final_first, "Final Accuracy on B0 Classes", "final_first_step_rank_runs.png")
make_barplot(final_later, "Final Accuracy on Later Classes", "final_later_steps_rank_runs.png")
make_barplot(final_seen,  "Final Accuracy on All Seen Classes", "final_all_seen_rank_runs.png")

In [ ]:
plot_df = results_df_clean.copy()
plot_df_step = plot_df[plot_df["eval_stage"] == "step"].copy()

for rank_value in RANK_LIST:
    method_name = f"LoRA-r{rank_value}"

    one_df = plot_df_step[plot_df_step["method"] == method_name].copy()
    if len(one_df) == 0:
        continue

    cur_df = one_df[one_df["eval_set"] == "current_step"].copy()
    first_df = one_df[one_df["eval_set"] == "first_step"].copy()
    seen_df = one_df[one_df["eval_set"] == "all_seen"].copy()

    plt.figure(figsize=(7, 4))
    plt.plot(cur_df["step"], cur_df["accuracy"], marker="o", label="current_step")
    plt.plot(first_df["step"], first_df["accuracy"], marker="o", label="first_step")
    plt.plot(seen_df["step"], seen_df["accuracy"], marker="o", label="all_seen")
    plt.xticks(cur_df["step"])
    plt.xlabel("Step")
    plt.ylabel("Accuracy")
    plt.title(f"{method_name}: Accuracy over steps")
    plt.legend()
    plt.tight_layout()

    out = os.path.join(PLOTS_DIR, f"curve_rank_{rank_value}_all_metrics.png")
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

In [ ]:
df_merge_compare = df_final[df_final["method"].isin([f"LoRA-r{r}" for r in RANK_LIST] + ["Merged across ranks"])].copy()

def make_barplot(df_sub, title, filename):
    methods = [f"LoRA-r{r}" for r in RANK_LIST] + ["Merged across ranks"]
    vals = []

    for m in methods:
        row = df_sub[df_sub["method"] == m]
        vals.append(float(row["accuracy"].iloc[0]) if len(row) > 0 else np.nan)

    plt.figure(figsize=(9, 4))
    plt.bar(methods, vals)
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.xticks(rotation=20)
    plt.tight_layout()

    out_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved plot to:", out_path)

final_first = df_merge_compare[df_merge_compare["eval_set"] == "first_step"]
final_later = df_merge_compare[df_merge_compare["eval_set"] == "later_steps"]
final_seen  = df_merge_compare[df_merge_compare["eval_set"] == "all_seen"]

make_barplot(final_first, "Ranks vs Final Merge on B0 Classes", "ranks_vs_merge_first_step.png")
make_barplot(final_later, "Ranks vs Final Merge on Later Classes", "ranks_vs_merge_later_steps.png")
make_barplot(final_seen,  "Ranks vs Final Merge on All Seen Classes", "ranks_vs_merge_all_seen.png")

In [ ]:
df_rank_only = df_final[df_final["method"].isin([f"LoRA-r{r}" for r in RANK_LIST] + ["Merged across ranks"])].copy()

pivot_rank = df_rank_only.pivot_table(
    index="eval_set",
    columns="method",
    values="accuracy",
    aggfunc="mean"
).reindex(["first_step", "later_steps", "all_seen"])

x = np.arange(3)
methods = [m for m in [f"LoRA-r{r}" for r in RANK_LIST] + ["Merged across ranks"] if m in pivot_rank.columns]
width = 0.8 / len(methods)

plt.figure(figsize=(10, 5))
for i, m in enumerate(methods):
    plt.bar(x + (i - (len(methods)-1)/2) * width, pivot_rank[m], width, label=m)

plt.xticks(x, ["first_step", "later_steps", "all_seen"])
plt.ylabel("Accuracy")
plt.title("Effect of Rank and Final Delta Merge")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "rank_and_merge_comparison.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

In [ ]:
rank_merge_summary = df_final[df_final["method"].isin([f"LoRA-r{r}" for r in RANK_LIST] + ["Merged across ranks"])].copy()
print(rank_merge_summary.sort_values(["eval_set", "method"]))